# Target Encoding Experiment

This notebook tests smoothed target encoding as an alternative to one-hot encoding for categorical variables in the subrogation prediction project.

The goal is to compare these models against the one-hot encoded baselines from `04_model_baseline.ipynb`:

- `logistic_regression_target_encoding_v1`
- `xgboost_target_encoding_v1`

Target encoding is learned from the training split only to avoid validation leakage.

## 1. Imports and project paths

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [ ]:
# If this notebook is inside a notebooks/ folder, this points to the project root.
# Otherwise, it uses the current working directory as the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
METRICS_PATH = REPORTS_DIR / "model_metrics.json"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Metrics path:", METRICS_PATH)

## 2. Load engineered training data

This notebook assumes `03_feature_engineering.ipynb` has already created `data/processed/train_features.csv`.

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

In [ ]:
# Basic target check
print(df["subrogation"].value_counts(normalize=True).rename("proportion"))
print("
Missing target values:", df["subrogation"].isna().sum())

## 3. Define features and target

We drop identifier columns because they should not be used as predictive features.

In [ ]:
TARGET_COL = "subrogation"
ID_COLS = ["claim_number"]

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Drop identifier columns if present
X = X.drop(columns=[col for col in ID_COLS if col in X.columns])

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Identify categorical columns for target encoding
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Number of categorical columns:", len(categorical_cols))
print("Number of numeric columns:", len(numeric_cols))

## 4. Train/validation split

The split happens before target encoding so validation target values are never used to create encoded features.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train positive rate:", round(y_train.mean(), 4))
print("y_val positive rate:", round(y_val.mean(), 4))

## 5. Smoothed target encoding functions

For validation data, each category is encoded using target statistics from the training split only.

For training data, this notebook uses out-of-fold target encoding. That prevents each training row from being encoded using its own target value, which reduces overfitting.

In [ ]:
def _smoothed_category_mapping(X_col, y, smoothing=20):
    """
    Create a smoothed category-to-target mapping for one column.

    Smoothed mean = (count * category_mean + smoothing * global_mean) / (count + smoothing)
    """
    global_mean = y.mean()

    stats = (
        pd.DataFrame({"category": X_col.astype(str), "target": y})
        .groupby("category")["target"]
        .agg(["mean", "count"])
    )

    smoothed_mean = (
        (stats["count"] * stats["mean"] + smoothing * global_mean)
        / (stats["count"] + smoothing)
    )

    return smoothed_mean.to_dict(), global_mean


def target_encode_train_validation(
    X_train,
    X_val,
    y_train,
    categorical_cols,
    smoothing=20,
    n_splits=5,
    random_state=42
):
    """
    Apply out-of-fold smoothed target encoding to training data and
    train-only smoothed target encoding to validation data.
    """
    X_train_encoded = X_train.copy()
    X_val_encoded = X_val.copy()
    y_train = y_train.copy()

    encoder_artifacts = {
        "categorical_cols": categorical_cols,
        "smoothing": smoothing,
        "global_mean": float(y_train.mean()),
        "mappings": {}
    }

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in categorical_cols:
        encoded_col = f"{col}_te"
        X_train_encoded[encoded_col] = np.nan

        # Out-of-fold encoding for training rows
        for train_idx, holdout_idx in skf.split(X_train, y_train):
            X_fold_train = X_train.iloc[train_idx]
            y_fold_train = y_train.iloc[train_idx]
            X_fold_holdout = X_train.iloc[holdout_idx]

            fold_mapping, fold_global_mean = _smoothed_category_mapping(
                X_fold_train[col],
                y_fold_train,
                smoothing=smoothing
            )

            X_train_encoded.iloc[
                holdout_idx,
                X_train_encoded.columns.get_loc(encoded_col)
            ] = (
                X_fold_holdout[col]
                .astype(str)
                .map(fold_mapping)
                .fillna(fold_global_mean)
            )

        # Final mapping using full training data for validation and future inference
        full_mapping, global_mean = _smoothed_category_mapping(
            X_train[col],
            y_train,
            smoothing=smoothing
        )

        X_val_encoded[encoded_col] = (
            X_val[col]
            .astype(str)
            .map(full_mapping)
            .fillna(global_mean)
        )

        encoder_artifacts["mappings"][col] = full_mapping

    # Drop original categorical columns after encoding
    X_train_encoded = X_train_encoded.drop(columns=categorical_cols)
    X_val_encoded = X_val_encoded.drop(columns=categorical_cols)

    return X_train_encoded, X_val_encoded, encoder_artifacts


def clean_feature_names(df):
    """Clean feature names for XGBoost compatibility."""
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("[", "(", regex=False)
        .str.replace("]", ")", regex=False)
        .str.replace("<", "less_than", regex=False)
    )
    return df

## 6. Apply target encoding

In [ ]:
X_train_te, X_val_te, target_encoder_artifacts = target_encode_train_validation(
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    categorical_cols=categorical_cols,
    smoothing=20,
    n_splits=5,
    random_state=42
)

X_train_te = clean_feature_names(X_train_te)
X_val_te = clean_feature_names(X_val_te)

target_encoder_artifacts["feature_columns"] = X_train_te.columns.tolist()

print("X_train_te:", X_train_te.shape)
print("X_val_te:", X_val_te.shape)

In [ ]:
# Confirm no categorical columns remain
remaining_categorical = X_train_te.select_dtypes(include=["object", "category"]).columns.tolist()
remaining_categorical

In [ ]:
# Confirm XGBoost-incompatible feature-name characters are gone
bad_cols = [
    col for col in X_train_te.columns
    if any(char in str(col) for char in ["[", "]", "<"])
]

bad_cols

In [ ]:
X_train_te.head()

## 7. Helper functions for evaluation and experiment logging

In [ ]:
def evaluate_binary_classifier(model, X_val, y_val):
    """Evaluate a binary classifier using default threshold 0.50."""
    y_pred = model.predict(X_val)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_prob = y_pred

    return {
        "accuracy": round(accuracy_score(y_val, y_pred), 4),
        "precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_val, y_pred, zero_division=0), 4),
        "f1_score": round(f1_score(y_val, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_val, y_prob), 4)
    }


def save_model_and_append_metrics(
    model,
    model_name,
    model_family,
    X_val,
    y_val,
    metrics_path=METRICS_PATH,
    models_dir=MODELS_DIR,
    notes=None,
    extra_artifacts=None
):
    """Save model artifact and append validation metrics to JSON experiment log."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    run_id = f"{timestamp}_{model_name}"

    model_path = models_dir / f"{run_id}.pkl"

    artifact = {
        "model": model,
        "feature_columns": X_val.columns.tolist()
    }

    if extra_artifacts is not None:
        artifact.update(extra_artifacts)

    joblib.dump(artifact, model_path)

    record = {
        "run_id": run_id,
        "run_date": datetime.now().isoformat(timespec="seconds"),
        "model_name": model_name,
        "model_family": model_family,
        "model_path": str(model_path),
        "metrics": evaluate_binary_classifier(model, X_val, y_val),
        "notes": notes
    }

    if metrics_path.exists():
        with open(metrics_path, "r") as f:
            metrics_log = json.load(f)
    else:
        metrics_log = []

    metrics_log.append(record)

    with open(metrics_path, "w") as f:
        json.dump(metrics_log, f, indent=4)

    return record

## 8. Logistic Regression with target encoding

Logistic Regression benefits from scaling, so this experiment uses a `StandardScaler` inside a pipeline.

In [ ]:
log_reg_te = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_te.fit(X_train_te, y_train)

log_reg_te_record = save_model_and_append_metrics(
    model=log_reg_te,
    model_name="logistic_regression_target_encoding_v1",
    model_family="LogisticRegression",
    X_val=X_val_te,
    y_val=y_val,
    notes="Logistic regression using out-of-fold smoothed target encoding for categorical variables.",
    extra_artifacts={"target_encoder": target_encoder_artifacts}
)

log_reg_te_record["metrics"]

In [ ]:
log_reg_te_preds = log_reg_te.predict(X_val_te)

print("Confusion matrix:")
print(confusion_matrix(y_val, log_reg_te_preds))
print("
Classification report:")
print(classification_report(y_val, log_reg_te_preds, zero_division=0))

## 9. XGBoost with target encoding

XGBoost does not need scaling. This experiment keeps the model simple so the main change being tested is the categorical encoding strategy.

In [ ]:
xgb_te = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_te.fit(X_train_te, y_train)

xgb_te_record = save_model_and_append_metrics(
    model=xgb_te,
    model_name="xgboost_target_encoding_v1",
    model_family="XGBClassifier",
    X_val=X_val_te,
    y_val=y_val,
    notes="XGBoost using out-of-fold smoothed target encoding for categorical variables.",
    extra_artifacts={"target_encoder": target_encoder_artifacts}
)

xgb_te_record["metrics"]

In [ ]:
xgb_te_preds = xgb_te.predict(X_val_te)

print("Confusion matrix:")
print(confusion_matrix(y_val, xgb_te_preds))
print("
Classification report:")
print(classification_report(y_val, xgb_te_preds, zero_division=0))

## 10. Compare recent model runs

This table reads the shared metrics log and compares the latest runs by model name.

In [ ]:
with open(METRICS_PATH, "r") as f:
    metrics_log = json.load(f)

metrics_df = pd.DataFrame([
    {
        "run_date": record["run_date"],
        "model_name": record["model_name"],
        "model_family": record["model_family"],
        **record["metrics"]
    }
    for record in metrics_log
])

latest_metrics = (
    metrics_df
    .sort_values("run_date")
    .groupby("model_name", as_index=False)
    .tail(1)
    .sort_values("f1_score", ascending=False)
)

latest_metrics

## 11. Notes

Interpret target encoding results against the one-hot encoded baseline:

- If F1 improves, target encoding is a useful replacement or candidate for the tuned model.
- If ROC-AUC improves but F1 does not, threshold tuning is likely the next best step.
- If both F1 and ROC-AUC decline, keep the one-hot baseline and move to threshold tuning or hyperparameter tuning.